# Week 6.1 Concepts: LangChain Framework Basics

Runnable, standard-library-only companion examples for each day's lesson.
The Day 1 and Day 2 cells simulate LangChain's `Runnable` / `|` composition
with a tiny local class so nothing here needs `pip install langchain` or an
API key -- the composition mechanics (each step's output feeds the next
step's input) match how `langchain_core.runnables.Runnable.__or__` really
works, just without the real package installed.

## Day 1: LangChain and the LCEL Pipe

LangChain's `|` operator composes `Runnable` objects into a `RunnableSequence`:
calling `.invoke()` on the sequence runs each step in order, feeding one
step's return value into the next step's input. Below we build a minimal
`MiniRunnable` base class that reproduces exactly that behavior, then wire
a template, a mock model, and a text-shaping function into a chain.

In [ ]:
class MiniRunnable:
    """Local stand-in for langchain_core.runnables.Runnable.
    Real LangChain Runnables expose the same shape: an .invoke() method
    and a __or__ that composes two steps into a sequence."""

    def invoke(self, value):
        raise NotImplementedError

    def __or__(self, other):
        return MiniSequence(self, other)


class MiniSequence(MiniRunnable):
    def __init__(self, first, second):
        self.first = first
        self.second = second

    def invoke(self, value):
        # This is the whole trick behind prompt | llm | parser:
        # the first step's output becomes the second step's input.
        return self.second.invoke(self.first.invoke(value))


class FuncStep(MiniRunnable):
    """Wraps a plain function as a pipeline step (LangChain does this
    coercion for you automatically when you pipe into a callable)."""

    def __init__(self, fn):
        self.fn = fn

    def invoke(self, value):
        return self.fn(value)


class TinyTemplate(MiniRunnable):
    """Local stand-in for PromptTemplate.from_template(...)."""

    def __init__(self, template):
        self.template = template

    def invoke(self, variables):
        return self.template.format(**variables)


In [ ]:
class StubForecaster(MiniRunnable):
    """Mock 'model' -- matches the .invoke() shape a real chat model would
    have, but returns a scripted string instead of calling any API."""

    def invoke(self, prompt_text):
        return f"[stub-forecast] based on '{prompt_text[:40]}...', expect steady demand"


weather_prompt = TinyTemplate("Given these signals: {signals}, forecast next week's sales.")
trim_and_shout = FuncStep(lambda text: text.strip().upper())

forecast_chain = weather_prompt | StubForecaster() | trim_and_shout

print(forecast_chain.invoke({"signals": "rising foot traffic, no promotions running"}))
print(forecast_chain.invoke({"signals": "holiday week, heavy discounting"}))


## Day 2: Reusable Templates, Swappable Models, Manual Parsing

A template is defined once and reused with different inputs. Swapping model
providers is mostly a construction-line change because every chat model
class implements the same call interface (schematic below -- it needs real
API keys to actually run). Then: parsing and validating raw model output
by hand, without any framework output-parser helper.

In [ ]:
advice_prompt = TinyTemplate("Give a one-line {tone} tip for a {role}.")

requests = [
    {"tone": "encouraging", "role": "first-time marathon runner"},
    {"tone": "blunt", "role": "new backend engineer"},
    {"tone": "encouraging", "role": "piano beginner"},
]
for r in requests:
    print(advice_prompt.invoke(r))

# Swapping providers is mostly a construction-line change -- both chat model
# classes below implement the same base interface, so the surrounding code
# (build prompt, call .invoke(), read .content) barely changes. This needs
# real installs + API keys to run, so it's shown schematically:
#
# from langchain_openai import ChatOpenAI
# from langchain_google_genai import ChatGoogleGenerativeAI
# model_a = ChatOpenAI(model="gpt-4o-mini")
# model_b = ChatGoogleGenerativeAI(model="gemini-1.5-flash")
# for model in (model_a, model_b):
#     print(model.invoke("Name one LCEL benefit in five words.").content)


In [ ]:
import json

def parse_and_validate_ticket(raw: str) -> dict:
    """Hand-rolled parsing + validation -- no framework output parser."""
    record = json.loads(raw)  # raises on malformed JSON

    required = {"priority", "category", "eta_minutes"}
    missing = required - record.keys()
    if missing:
        raise ValueError(f"missing fields: {missing}")

    if record["priority"] not in {"low", "medium", "high"}:
        raise ValueError(f"bad priority: {record['priority']}")

    eta = record["eta_minutes"]
    if not isinstance(eta, int) or isinstance(eta, bool) or eta <= 0:
        raise ValueError(f"eta_minutes must be a positive int, got {eta!r}")

    return record


raw_outputs = [
    '{"priority": "high", "category": "billing", "eta_minutes": 15}',
    '{"priority": "medium", "category": "login", "eta_minutes": 30}',
]
for raw in raw_outputs:
    print(parse_and_validate_ticket(raw))

# A malformed record correctly raises instead of silently passing through:
bad_raw = '{"priority": "urgent", "category": "billing", "eta_minutes": 15}'
try:
    parse_and_validate_ticket(bad_raw)
except ValueError as e:
    print("correctly rejected:", e)


## Day 3: A Safe Calculator Tool and an FAQ Tool

`eval()` is unsafe for user-supplied expressions. `ast.literal_eval` is
*also* not enough for a calculator: it only parses literal constants and
containers, not an arithmetic operation between two literals --
`ast.literal_eval("6 * 7")` raises `ValueError` because multiplication
isn't part of its restricted grammar. The fix is a custom AST walker that
whitelists arithmetic nodes and rejects everything else (names, calls,
attribute access, subscripts, ...).

In [ ]:
import ast
import operator

_ALLOWED_BIN_OPS = {
    ast.Add: operator.add,
    ast.Sub: operator.sub,
    ast.Mult: operator.mul,
    ast.Div: operator.truediv,
    ast.Pow: operator.pow,
}
_ALLOWED_UNARY_OPS = {
    ast.UAdd: operator.pos,
    ast.USub: operator.neg,
}


def _eval_node(node):
    if isinstance(node, ast.Expression):
        return _eval_node(node.body)

    if isinstance(node, ast.Constant):
        if isinstance(node.value, (int, float)) and not isinstance(node.value, bool):
            return node.value
        raise ValueError(f"unsupported constant: {node.value!r}")

    # ast.Num is deprecated (Python 3.8+) but kept here for older parse trees.
    if hasattr(ast, "Num") and isinstance(node, ast.Num):
        return node.n

    if isinstance(node, ast.BinOp):
        op_type = type(node.op)
        if op_type not in _ALLOWED_BIN_OPS:
            raise ValueError(f"operator not allowed: {op_type.__name__}")
        return _ALLOWED_BIN_OPS[op_type](_eval_node(node.left), _eval_node(node.right))

    if isinstance(node, ast.UnaryOp):
        op_type = type(node.op)
        if op_type not in _ALLOWED_UNARY_OPS:
            raise ValueError(f"unary operator not allowed: {op_type.__name__}")
        return _ALLOWED_UNARY_OPS[op_type](_eval_node(node.operand))

    raise ValueError(f"unsupported expression: {type(node).__name__}")


def calc(expr: str):
    """Safely evaluate a plain arithmetic expression. No names, no calls,
    no attribute access -- only numbers and +, -, *, /, ** between them."""
    parsed = ast.parse(expr, mode="eval")
    return _eval_node(parsed)


for expr in ["6 * 7", "(9 - 3) ** 2 / 4", "-8 + 20 / 4", "3 ** 4 - 1"]:
    print(expr, "=>", calc(expr))

for bad_expr in ["__import__('os').system('ls')", "(1).__class__", "open('secrets.txt')", "[1, 2, 3]"]:
    try:
        calc(bad_expr)
        print("SHOULD NOT REACH HERE:", bad_expr)
    except ValueError as e:
        print("correctly rejected:", bad_expr, "->", e)

# ast.literal_eval genuinely cannot do this -- confirms the claim above:
try:
    ast.literal_eval("6 * 7")
except ValueError as e:
    print("literal_eval correctly refuses '6 * 7':", e)


In [ ]:
_LIBRARY_FAQ = [
    (("renew", "extend"), "Loans can be renewed twice online before the due date."),
    (("overdue", "late fee"), "Late fees are $0.25/day per item, capped at $10."),
    (("hours", "open", "close"), "The branch is open 10am-8pm on weekdays, 10am-5pm weekends."),
]

def library_faq(question: str):
    q = question.lower()
    for keywords, answer in _LIBRARY_FAQ:
        if any(kw in q for kw in keywords):
            return answer
    return None


def route(user_input: str) -> str:
    has_digit = any(c.isdigit() for c in user_input)
    has_operator = any(op in user_input for op in "+-*/")
    if has_digit and has_operator:
        try:
            return f"calc tool => {calc(user_input)}"
        except ValueError as e:
            return f"calc tool refused: {e}"
    answer = library_faq(user_input)
    if answer:
        return f"faq tool => {answer}"
    return "fallback => no matching tool"


sample_inputs = [
    "9 * 6",
    "can I renew my loan?",
    "what are your hours",
    "what is nine times six",  # known limitation: no digits/operators -> misrouted
]
for text in sample_inputs:
    print(f"{text!r:35} -> {route(text)}")


## Day 4: Memory, Growth, and Persistence

Each model call is stateless -- "memory" means re-sending prior turns as
part of the next prompt, so an unbounded conversation makes every later
prompt bigger (and slower, and costlier). Two bounding strategies: keep a
recent window, or periodically fold older turns into a summary. Persisting
history (e.g. to SQLite) lets a conversation survive a restart -- but
anything a user typed, PII included, is stored verbatim unless you actively
redact it before writing, which matters for data-retention/compliance.

In [ ]:
def render_prompt(history, new_message):
    convo = "\n".join(f"{speaker}: {text}" for speaker, text in history)
    return f"{convo}\nuser: {new_message}\nassistant:"


notes_history = []
for turn in range(1, 16):
    msg = f"add agenda item {turn} to today's meeting notes"
    prompt = render_prompt(notes_history, msg)
    notes_history.append(("user", msg))
    notes_history.append(("assistant", f"added item {turn}"))
    if turn % 5 == 0:
        print(f"turn {turn}: prompt is {len(prompt)} characters")


def windowed(history, max_turns=8):
    """Mitigation 1: keep only the most recent N turns."""
    return history[-max_turns:]


def summarized(history, keep_recent=8, summarize=None):
    """Mitigation 2: collapse older turns into one running summary line."""
    if len(history) <= keep_recent:
        return history
    old, recent = history[:-keep_recent], history[-keep_recent:]
    old_text = "\n".join(f"{s}: {t}" for s, t in old)
    summary = summarize(old_text) if summarize else old_text[:120] + "..."
    return [("system", f"summary of earlier turns: {summary}")] + list(recent)


print("windowed turns kept:", len(windowed(notes_history)))
print("summarized turns kept:", len(summarized(notes_history)))


In [ ]:
import sqlite3

def save_history_sqlite(history, db_path="conversations.db", conversation_id="demo"):
    """Persist history so it survives a process restart."""
    conn = sqlite3.connect(db_path)
    conn.execute(
        "CREATE TABLE IF NOT EXISTS turns (conversation_id TEXT, turn_index INTEGER, "
        "speaker TEXT, text TEXT)"
    )
    conn.execute("DELETE FROM turns WHERE conversation_id = ?", (conversation_id,))
    conn.executemany(
        "INSERT INTO turns VALUES (?, ?, ?, ?)",
        [(conversation_id, i, speaker, text) for i, (speaker, text) in enumerate(history)],
    )
    conn.commit()
    conn.close()


def load_history_sqlite(db_path="conversations.db", conversation_id="demo"):
    conn = sqlite3.connect(db_path)
    conn.execute(
        "CREATE TABLE IF NOT EXISTS turns (conversation_id TEXT, turn_index INTEGER, "
        "speaker TEXT, text TEXT)"
    )
    rows = conn.execute(
        "SELECT speaker, text FROM turns WHERE conversation_id = ? ORDER BY turn_index",
        (conversation_id,),
    ).fetchall()
    conn.close()
    return rows


# PII note: if a turn contains something like "my email is a.kim@example.com",
# that string lands in the turns table exactly as typed -- sqlite does not
# know or care that it's sensitive. Redact or drop such fields before calling
# save_history_sqlite in a real system; this matters for data-retention and
# compliance reasons, not just tidiness.

save_history_sqlite(notes_history[:4], db_path="/tmp/demo_conversations.db")
print(load_history_sqlite(db_path="/tmp/demo_conversations.db"))
